In [1]:
import pandas as pd
import pyterrier as pt
import numpy as np
from pyterrier_t5 import MonoT5ReRanker
from pyterrier_pisa import PisaIndex

/opt/miniconda3/envs/inpars/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
training_docs = pd.read_pickle("/nfs/primary/sas_reranker/d2qmm_ohsumed_training_data.pkl")

In [3]:
training_docs = training_docs.rename(columns = {"medline_ui" : "docno"})

In [4]:
training_docs

,docno,title,abstract,sensitivity,text,querygen,querygen_score
0,88000001,The binding of acetaldehyde to the active site...,"Ribonuclease A was reacted with [1-13C,1,2-14C...",0,The binding of acetaldehyde to the active site...,what site of activity is used to bind ribonucl...,"[0.88141286, 2.3967025, 0.32945463, 0.9057854,..."
1,88000002,Reductions in breath ethanol readings in norma...,Blood ethanol concentrations were measured seq...,0,Reductions in breath ethanol readings in norma...,is ethanol in breath normal\nis blood ethanol ...,"[2.034942, 3.5010455, 0.3425358, 0.5842041, 0...."
2,88000003,Does the blockade of opioid receptors influenc...,We have tested whether the opioid antagonists ...,0,Does the blockade of opioid receptors influenc...,which opioids can cause ethanol dependence?\ne...,"[2.229129, 1.2757981, 2.7790852, 0.18533784, 2..."
3,88000006,Drinkwatchers--description of subjects and eva...,Clinical examination and measurement of MCV an...,0,Drinkwatchers--description of subjects and eva...,heavy drinking blood test\nhow much alcohol is...,"[0.87555873, 0.48380333, 1.1801778, 0.89115256..."
4,88000007,Platelet affinity for serotonin is increased i...,The kinetics of 3H serotonin platelet uptake w...,0,Platelet affinity for serotonin is increased i...,what is the biological marker for dependence\n...,"[3.0765808, 1.4992186, 1.945047, 1.81894, 2.75..."
...,...,...,...,...,...,...,...
334129,87325749,Clinical investigation of Duchenne muscular dy...,Between 1979 and 1987 we documented the natura...,0,Clinical investigation of Duchenne muscular dy...,what is clinical investigation of duchenne mus...,"[4.7701135, 3.1781783, 3.1509128, 3.3405652, 3..."
334130,87325791,Prostaglandins. Old concepts and new perspecti...,Lupus anticoagulants and anticardiolipin antib...,0,Prostaglandins. Old concepts and new perspecti...,what prostaglandins do\nprostaglandins lupus\n...,"[0.8321806, 1.5327109, 3.6516025, 2.132977, 0...."
334131,87325811,Vinculin in focal cell-to-substrate attachment...,Vinculin is a specialized protein with a molec...,0,Vinculin in focal cell-to-substrate attachment...,what is the function of vinculin\nwhat is vinc...,"[1.5229188, 3.623153, 3.2570174, 2.759778, 1.5..."
334132,87325849,Intraocular fibrinolysis with recombinant huma...,Intraocular fibrin formation following ocular ...,0,Intraocular fibrinolysis with recombinant huma...,recombinant plasminogen activator treatment\ni...,"[1.6216362, 3.4744203, 3.2590685, 1.8076218, 0..."


In [5]:
def get_best_query(row):
    queries = row['querygen'].split('\n')
    scores = row['querygen_score']
    if scores.size != 0:
        index = np.argmax(scores)
        best_query = queries[index]
    else:
        best_query = "query"
    return best_query

training_docs['best_query'] = training_docs.apply(get_best_query, axis=1)

# Create the new dataframe with queryid and best query
queries = pd.DataFrame({
    'qid': range(len(training_docs)),
    'query': training_docs['best_query']
})

queries

,qid,query
0,0,does acetaldehyde react with ribonuclease A
1,1,is blood ethanol measured sequentially for a n...
2,2,does the blockade of opioid receptors influenc...
3,3,how much alcohol do drinkwatchers drink
4,4,does alcoholics use serotonin to increase plat...
...,...,...
334129,334129,what is clinical investigation of duchenne mus...
334130,334130,what is a lupus anticoagulant and anticardioli...
334131,334131,where does vinculin attach in cells to a subst...
334132,334132,intraocular fibrinolysis with recombinant plas...


In [6]:
index = PisaIndex("./indices/ohsumed_training.pisa/")
bm25 = index.bm25(verbose = True, threads = 16)

In [7]:
results = bm25(queries)

PISA bm25: 100%|██████████| 334134/334134 [00:37<00:00, 8804.25query/s]


In [8]:
# pt.io.write_results(results, "bm25_top_1k.res")

In [9]:
def get_text(run):
    training_docs.docno = training_docs.docno.astype(str)
    new_run = pd.merge(training_docs, run, on = "docno")
    assert len(new_run) == len(run)
    return new_run

monot5 = MonoT5ReRanker()
pipeline = results >> pt.apply.generic(get_text) >> monot5

/opt/miniconda3/envs/inpars/lib/python3.10/site-packages/huggingface_hub/file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/opt/miniconda3/envs/inpars/lib/python3.10/site-packages/transformers/models/t5/tokenization_t5.py:240: FutureWarning: This tokenizer was incorrectly instantiated with a model max length of 512 which will be corrected in Transformers v5.
For now, this behavior is kept to avoid breaking backwards compatibility when padding/encoding with `truncation is True`.
- Be aware that you SHOULD NOT rely on t5-base automatically truncating your input to 512 when padding/encoding.
- If you want to encode/pad to sequences longer than 512 you can either instantiate this tokenizer with `model_max_length` or pass `max_length` when encoding/padding.
- To avoid this warning, please instantiate this tokenizer w

In [10]:
pipeline(queries)

KeyboardInterrupt: 